In [1]:
import google.genai as genai
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
gemini_key=os.getenv('GEMINI_API_KEY')

In [3]:
client = genai.Client()

In [4]:
response = client.models.generate_content(
    model="gemini-2.5-flash", contents="What's happening in Myanmar in 10 words."
)

In [5]:
print(response.text)

Military coup, brutal crackdown, widespread resistance, escalating civil war, dire crisis.


In [6]:
SYSTEM_PROMPT = """You are an expert PostgreSQL database assistant for an e-commerce analytics system.

Your task is to convert natural language questions into valid PostgreSQL queries that can be executed directly.

DATABASE SCHEMA:

Table: customers
- customer_id (VARCHAR, PRIMARY KEY) - Unique customer identifier
- customer_unique_id (VARCHAR) 
- customer_zip_code_prefix (INTEGER) - Zip code
- customer_city (VARCHAR) - City name
- customer_state (VARCHAR) - Two-letter state code (e.g., 'SP', 'RJ')

Table: orders
- order_id (VARCHAR, PRIMARY KEY) - Unique order identifier
- customer_id (VARCHAR, FOREIGN KEY → customers.customer_id)
- order_status (VARCHAR) - Status: delivered, shipped, canceled, etc.
- order_purchase_timestamp (TIMESTAMP) - When order was placed
- order_approved_at (TIMESTAMP) - When order was approved
- order_delivered_carrier_date (TIMESTAMP) - When shipped
- order_delivered_customer_date (TIMESTAMP) - When delivered to customer
- order_estimated_delivery_date (TIMESTAMP) - Estimated delivery date

Table: products
- product_id (VARCHAR, PRIMARY KEY) - Unique product identifier
- product_category_name (VARCHAR) - Category name in Portuguese
- product_name_lenght (INTEGER) - Product name length
- product_description_lenght (INTEGER) - Product Description length
- product_photos_qty (INTEGER) - Product photo qty
- product_weight_g (INTEGER) - Weight in grams
- product_length_cm (INTEGER) - Length in cm
- product_height_cm (INTEGER) - Height in cm
- product_width_cm (INTEGER) - Width in cm

Table: product_category_name_translation
- id (INTEGER, PRIMARY KEY) - Unique id
- product_category_name (VARCHAR) - Portuguese category name
- product_category_name_english (VARCHAR) - English translation

Table: order_items
- order_id (TEXT, FOREIGN KEY → orders.order_id)
- order_item_id (INTEGER) - Item sequence number within order
- product_id (TEXT, FOREIGN KEY → products.product_id)
- seller_id (TEXT) - Seller identifier
- shipping_limit_date (TIMESTAMP) - Shipping deadline
- price (NUMERIC) - Item price in Brazilian Reais
- freight_value (NUMERIC) - Shipping cost
- PRIMARY KEY: (order_id, order_item_id)

Table: order_payments
- order_id (TEXT, FOREIGN KEY → orders.order_id)
- payment_sequential (INTEGER) - Payment installment number
- payment_type (TEXT) - credit_card, boleto, voucher, debit_card
- payment_installments (INTEGER) - Number of installments
- payment_value (NUMERIC) - Payment amount
- PRIMARY KEY: (order_id, payment_sequential)

Table: order_reviews
- review_id (TEXT, PRIMARY KEY) - Unique review identifier
- order_id (TEXT, FOREIGN KEY → orders.order_id)
- review_score (INTEGER) - Rating from 1 to 5 stars
- review_comment_title (TEXT) - Review title
- review_comment_message (TEXT) - Review text
- review_creation_date (TIMESTAMP) - When review was created
- review_answer_timestamp (TIMESTAMP) - When seller responded

CRITICAL RULES:
1. Return ONLY the SQL query - no explanations, no markdown, no preamble
2. Do NOT use DROP, DELETE, UPDATE, ALTER, TRUNCATE, or INSERT statements
3. Always use proper JOIN syntax when combining tables
4. To get English product category names, JOIN products with product_category_name_translation on product_category_name
5. Use LIMIT to restrict large result sets (default LIMIT 100 if not specified)
6. All dates are stored as TIMESTAMP - use proper date functions for filtering
7. For "revenue" or "sales", calculate: SUM(price + freight_value) from order_items
8. Use table aliases for readability (e.g., 'c' for customers, 'o' for orders)

EXAMPLES:

Question: How many customers are there?
SQL: SELECT COUNT(*) as customer_count FROM customers;

Question: What are the top 5 product categories by number of orders?
SQL: SELECT pct.product_category_name_english, COUNT(DISTINCT oi.order_id) as order_count
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN product_category_name_translation pct ON p.product_category_name = pct.product_category_name
GROUP BY pct.product_category_name_english
ORDER BY order_count DESC
LIMIT 5;

Question: Show me 5 customers from Rio de Janeiro
SQL: SELECT customer_id, customer_city, customer_state
FROM customers
WHERE customer_state = 'RJ'
LIMIT 100;

Question: What is the total revenue in 2024?
SQL: SELECT ROUND(SUM(oi.price + oi.freight_value)::numeric,2) as total_revenue
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2024;

Question: Which orders have 5-star reviews?
SQL: SELECT o.order_id, o.customer_id, o.order_purchase_timestamp, r.review_score
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE r.review_score = 5
LIMIT 100;

Now convert this question into a SQL query:

Remark: Timestamp data are between 2024 and 2026. When user prompt full city name remember to change to lowercase.

Question: {question}
SQL:"""


def get_prompt(user_question: str) -> str:
    """
    Insert user question into the system prompt template.
    
    Args:
        user_question: The natural language question from the user
        
    Returns:
        Complete prompt ready to send to Gemini
    """
    return SYSTEM_PROMPT.format(question=user_question)

In [27]:
user_questions = "What were the top 3 product categories by number of orders?"

In [31]:
import time

logging_dict = {}
logging_dict['timestamp_value'] = time.time()
logging_dict['user_questions'] = user_questions
logging_dict['execution_time'] = 0
logging_dict['error'] = None

print(logging_dict)

{'timestamp_value': 1773408396.3616052, 'user_questions': 'What were the top 3 product categories by number of orders?', 'execution_time': 0, 'error': None}


In [32]:

prompt = get_prompt(user_questions)
try:
    start_time = time.time()
    response = client.models.generate_content(
    model="gemini-2.5-flash", contents=prompt
    )
    end_time = time.time()
    logging_dict['execution_time'] = end_time - start_time
    logging_dict['status'] = 'success'
except Exception as e:
    print(f"Error: {e}")
    logging_dict['error'] = e
    logging_dict['status'] = 'failed'

In [34]:
sql = response.text
print(sql)
print(logging_dict)

SELECT pct.product_category_name_english, COUNT(DISTINCT oi.order_id) AS order_count
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN product_category_name_translation pct ON p.product_category_name = pct.product_category_name
GROUP BY pct.product_category_name_english
ORDER BY order_count DESC
LIMIT 3;
{'timestamp_value': 1773408396.3616052, 'user_questions': 'What were the top 3 product categories by number of orders?', 'execution_time': 2.4683539867401123, 'error': None, 'status': 'success'}


In [ ]:
import pandas as pd
df = pd.DataFrame([logging_dict])

,timestamp_value,user_questions,execution_time,error,status
0,1.773408e+09,What were the top 3 product categories by numb...,2.468354,None,success


In [9]:
from sqlalchemy import create_engine, text
import pandas as pd

In [10]:
if sql == None:
    print("❌ Could not generate SQL. Your question might be ambiguous.")
    print("Please provide more details about which tables or columns you're interested in.")
else:
    dangerous_keywords = ['DROP', 'DELETE', 'UPDATE', 'ALTER', 'TRUNCATE', 'INSERT']
    if any(keyword in sql.upper() for keyword in dangerous_keywords):
        print("🚫 Error: Destructive queries are not allowed.")
        print("Only SELECT queries are permitted.")
    else: 
        dbname=os.getenv('DB_NAME')
        user=os.getenv('DB_USER')
        password=os.getenv('DB_PASS')
        host=os.getenv('DB_HOST')
        port=os.getenv('DB_PORT')

        engine = create_engine(
            f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
        )
        try: 
            with engine.connect() as conn:
                conn.execute(text("SET statement_timeout = '5s'"))
                df = pd.read_sql(text(sql), conn)
                if not df.empty:
                    print("\nQuery Results:")
                    print(df.to_string(index=False))
                    print(f"\nReturned {len(df)} row(s)")
                else:
                    print("Query executed successfully but returned no results.")
        except Exception as e:
            if "canceling statement due to statement timeout" in str(e):
                print("⏱️ Query timeout: Query took longer than 5 seconds.")
                print("Try narrowing your search with date filters or LIMIT.")
            else:
                print(f"SQL Execution Error: {e}")
                print(f"\nGenerated SQL was:\n{sql}")


Query Results:
product_category_name_english  order_count
               bed_bath_table         9417
                health_beauty         8836
               sports_leisure         7720

Returned 3 row(s)


In [11]:
df

,product_category_name_english,order_count
0,bed_bath_table,9417
1,health_beauty,8836
2,sports_leisure,7720


1773407118.828575

2023-11-25 01:00:00
